# Tutorial 3-2: K-Nearest Neighbors Decision Boundaries
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

## Objective
The K-Nearest Neighbor (KNN) algorithm is one of the simplest yet most flexible classification tools in machine learning. It is an **Instance-Based Learner**, meaning it doesn't build an explicit model; instead, it memorizes the training data and performs 'work' only when a new, unknown record needs a label. In this tutorial, we will explore:
1. **The Decision Landscape:** How KNN carves out complex boundaries based on local proximity.
2. **Hyperparameter Tuning:** Analyzing how the choice of $k$ dictates the balance between noise sensitivity and generalization.
3. **Distance Weighting:** Enhancing the 'vote' of closer neighbors to improve accuracy.
4. **The Curse of Dimensionality:** Understanding why high-dimensional data makes distance-based logic fail.

## 1. Visualizing Non-Linear Boundaries
Unlike linear classifiers that try to draw a straight line between classes, KNN can produce arbitrarily shaped decision boundaries. This makes it powerful for datasets where classes are intertwined. 

We will use the **Moons dataset**—a classic synthetic dataset where two classes are shaped like interlocking crescents.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier
from scipy.spatial import distance 

# 1. Generate the Moons dataset
X, y = make_moons(n_samples=300, noise=0.25, random_state=42)

def plot_decision_regions(X, y, k_value, weight_type='uniform'):
    # Initialize and fit the KNN classifier
    knn = KNeighborsClassifier(n_neighbors=k_value, weights=weight_type)
    knn.fit(X, y)
    
    # Define the grid for plotting
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
    
    # Predict labels for every point on the grid
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    # Plot the results
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', s=40, cmap='coolwarm')
    plt.title(f"KNN Decision Boundary (k={k_value}, weights='{weight_type}')")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

# Visualizing a single neighbor (Local logic)
plot_decision_regions(X, y, k_value=1)

## 2. Choosing the Optimal K
The value of $k$ is critical. It determines the size of the neighborhood used for the majority vote. 

- **Small $k$ (e.g., $k=1$):** High variance. The boundary is 'jagged' and highly sensitive to noise points. This is likely to overfit.
- **Large $k$ (e.g., $k=100$):** High bias. The boundary becomes very smooth, but it may start including points from far-away classes, causing the model to miss the nuances of the data shape.

In [ ]:
plt.figure(figsize=(15, 5))

for i, k in enumerate([1, 10, 100]):
    plt.subplot(1, 3, i + 1)
    # Note: We repeat the logic inside the loop for visual clarity in subplots
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', s=20, cmap='coolwarm')
    plt.title(f"k = {k}")

plt.tight_layout()
plt.show()

## 3. Weighted KNN
In standard KNN, every neighbor has an equal vote. However, a neighbor that is very close to the unknown record should probably have more influence than a neighbor that is further away. 

By setting `weights='distance'`, we weight each vote by the inverse of the distance, $w = 1/d$. This often produces a more robust boundary that respects local density.

In [ ]:
plot_decision_regions(X, y, k_value=15, weight_type='distance')

## 4. The Curse of Dimensionality
As we discussed in class, as the number of dimensions (features) increases, data becomes increasingly sparse. The concept of 'nearest' begins to lose meaning because the distance between the closest point and the farthest point converges. 

Let's visualize how the relative difference between max and min distances shrinks as we add more dimensions to our data.

In [ ]:
from scipy.spatial import distance # Ensuring the package is available for this cell

dimensions = range(1, 101, 5)
dist_ratios = []

for d in dimensions:
    # Generate 500 random points in d-dimensional space
    points = np.random.rand(500, d)
    
    # Compute all-to-all Euclidean distances
    dist_matrix = distance.cdist(points, points, metric='euclidean')
    
    # Ignore distances to self (diagonal zeros)
    non_zero_distances = dist_matrix[dist_matrix > 0]
    
    min_dist = np.min(non_zero_distances)
    max_dist = np.max(non_zero_distances)
    
    # Calculate the ratio as discussed in Slide 39
    ratio = (max_dist - min_dist) / min_dist
    dist_ratios.append(np.log10(ratio))

plt.figure(figsize=(10, 6))
plt.plot(dimensions, dist_ratios, marker='o', linestyle='--', color='darkblue')
plt.title("The Curse of Dimensionality: Distance Ratio Decay")
plt.xlabel("Number of Dimensions")
plt.ylabel("log10((Max_Dist - Min_Dist) / Min_Dist)")
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## Summary
K-Nearest Neighbor classification is powerful because it makes no assumptions about the distribution of the data. However, as we have seen:
1. **Choice of $k$** is a trade-off between sensitivity to noise and missing local patterns.
2. **Weighting** can significantly improve the logic of the majority vote.
3. **Feature Scaling** (from previous tutorials) is mandatory, as KNN relies entirely on distance metrics.
4. **Dimensionality Reduction** is often necessary for high-dimensional datasets to maintain the 'meaning' of proximity.